<center><strong>Deep Learning : Computer Vision</strong><br />
<strong><font color="blue">Departemen Informatika - Prodi RKA - Institut Teknologi Sepuluh Nopember</font></strong><br />
</center>

<strong>Outline Hands-On: Image Classification with CNN</strong><br />
<li> Pengenalan Computer Vision & Image Classification </li>
<li> Convolutional Neural Network (CNN) </li>
<li> Implementasi CNN sederhana dengan dataset MNIST </li>
<li> Implementasi CNN dengan dataset CIFAR-10 </li>
<li> Mengatasi Overfitting: Dropout & Data Augmentation </li>
<li> Transfer Learning dengan Pre-trained Model </li>
<li> Fine-Tuning Pre-trained Model </li>

---

## Setup & Persiapan Environment

<p><strong><font color="red">PENTING:</font></strong> Pastikan runtime Google Colab menggunakan <strong>GPU</strong> untuk mempercepat proses training.</p>

<p>Cara mengaktifkan GPU di Google Colab:</p>
<ol>
    <li>Klik menu <strong>Runtime</strong> → <strong>Change runtime type</strong></li>
    <li>Pada <strong>Hardware accelerator</strong>, pilih <strong>GPU (T4)</strong></li>
    <li>Klik <strong>Save</strong></li>
</ol>

In [ ]:
# Import library yang dibutuhkan
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

---
## 1. Pengenalan Computer Vision & Image Classification

<p><strong>Computer Vision</strong> adalah cabang dari Artificial Intelligence yang memungkinkan komputer untuk <em>meniru kemampuan manusia</em> dalam melihat dan memahami informasi visual.</p>

<p>Beberapa task utama dalam Computer Vision:</p>
<ul>
    <li><strong>Image Classification</strong> — Menentukan kategori/kelas dari suatu gambar (contoh: gambar kucing atau anjing)</li>
    <li><strong>Object Detection</strong> — Mendeteksi dan melokalisasi objek dalam gambar dengan bounding box</li>
    <li><strong>Instance Segmentation</strong> — Segmentasi piksel per-objek secara individual</li>
    <li><strong>Image Captioning</strong> — Menghasilkan deskripsi teks dari gambar</li>
</ul>

<p>Pada hands-on ini kita akan fokus pada <strong>Image Classification</strong>.</p>

### Machine Learning vs Deep Learning untuk Image Classification

<p>Ada dua pendekatan utama:</p>

<table border="1" cellpadding="8" cellspacing="0">
<tr><th></th><th><strong>Machine Learning (Tradisional)</strong></th><th><strong>Deep Learning (CNN)</strong></th></tr>
<tr><td><strong>Feature Extraction</strong></td><td>Manual/Hand-crafted (HOG, LBP, GLCM, dll)</td><td>Otomatis dipelajari oleh model</td></tr>
<tr><td><strong>Pipeline</strong></td><td>Feature Extraction → Classifier terpisah</td><td>End-to-end (extraction + classification sekaligus)</td></tr>
<tr><td><strong>Data</strong></td><td>Bisa bekerja dengan data kecil</td><td>Butuh data besar untuk performa optimal</td></tr>
<tr><td><strong>Performa</strong></td><td>Baik untuk domain spesifik</td><td>Umumnya lebih unggul di berbagai task</td></tr>
</table>

---
## 2. Convolutional Neural Network (CNN)

<p><strong>CNN (Convolutional Neural Network)</strong> adalah arsitektur neural network yang dirancang khusus untuk memproses data visual. CNN belajar <strong>pola lokal</strong> (local patterns) dari gambar, berbeda dengan Feedforward NN yang belajar pola global.</p>

<p><strong>Karakteristik utama CNN:</strong></p>
<ul>
    <li><strong>Translation Invariant</strong> — Pola yang dipelajari tidak bergantung pada posisi di gambar</li>
    <li><strong>Spatial Hierarchy</strong> — CNN dapat mempelajari hierarki pola, dari fitur sederhana (edge, texture) ke fitur kompleks (objek, wajah)</li>
</ul>

### Komponen Utama Arsitektur CNN:

<ol>
    <li><strong>Convolutional Layer</strong> — Filter/kernel yang melakukan sliding pada gambar untuk mendeteksi fitur</li>
    <li><strong>Activation Layer (ReLU)</strong> — Menambahkan non-linearitas</li>
    <li><strong>Pooling Layer</strong> — Mengurangi dimensi (downsampling) dan mengekstrak fitur dominan</li>
    <li><strong>Dropout Layer</strong> — Regularisasi untuk mencegah overfitting</li>
    <li><strong>Fully Connected (Dense) Layer</strong> — Layer klasifikasi di bagian akhir</li>
</ol>

### Operasi Konvolusi

<p>Konvolusi adalah proses menggeser (<em>sliding</em>) sebuah filter/kernel di atas gambar. Filter berperan sebagai <strong>feature detector</strong>.</p>

<p><strong>Rumus ukuran output:</strong></p>
<p style="text-align:center; font-size:16px;"><code>Output Size = (N - F) / S + 1</code></p>
<p>Dimana: N = ukuran input, F = ukuran filter, S = stride</p>

<p><strong>Padding:</strong></p>
<ul>
    <li><strong>Valid padding</strong> — Tidak ada padding, ukuran output mengecil</li>
    <li><strong>Same padding</strong> — Ditambahkan zero padding agar ukuran output = ukuran input</li>
</ul>

<p><strong>Pooling Layer:</strong> Bertujuan untuk mengurangi dimensi dan mengekstrak fitur dominan. Tipe yang umum: <strong>Max Pooling</strong> (mengambil nilai maksimum) dan <strong>Average Pooling</strong>.</p>

Mari kita lihat bagaimana konvolusi bekerja secara visual:

In [ ]:
# Demonstrasi operasi konvolusi sederhana
import numpy as np

# Contoh gambar 5x5
image = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0]
])

# Filter 3x3 (edge detector)
filter_kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

# Konvolusi manual (valid padding, stride=1)
output_size = image.shape[0] - filter_kernel.shape[0] + 1  # (5-3)/1 + 1 = 3
output = np.zeros((output_size, output_size))

for i in range(output_size):
    for j in range(output_size):
        region = image[i:i+3, j:j+3]
        output[i, j] = np.sum(region * filter_kernel)

print("Input Image (5x5):")
print(image)
print(f"\nFilter (3x3):")
print(filter_kernel)
print(f"\nOutput (3x3):")
print(output)
print(f"\nOutput size: ({output_size}x{output_size}) = (N-F)/S + 1 = (5-3)/1 + 1 = 3")

In [ ]:
# Visualisasi konvolusi
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Input Image (5x5)', fontsize=12)
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, str(image[i,j]), ha='center', va='center', color='red', fontsize=12)

axes[1].imshow(filter_kernel, cmap='RdBu')
axes[1].set_title('Filter/Kernel (3x3)', fontsize=12)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(filter_kernel[i,j]), ha='center', va='center', color='black', fontsize=14)

axes[2].imshow(output, cmap='gray')
axes[2].set_title('Convolved Feature (3x3)', fontsize=12)
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, str(int(output[i,j])), ha='center', va='center', color='red', fontsize=12)

plt.tight_layout()
plt.show()

<h3><font color="blue">Latihan 1:</font></h3>

<ul>
    <li>Ubah filter menjadi horizontal edge detector: <code>[[1,1,1],[0,0,0],[-1,-1,-1]]</code>. Apa perbedaan hasilnya?</li>
    <li>Apa yang terjadi jika stride diubah menjadi 2? Hitung ukuran output secara manual terlebih dahulu.</li>
</ul>

In [ ]:
# Kerjakan Latihan 1 pada cell berikut ini



---
## 3. CNN untuk Klasifikasi MNIST

<p>Dataset <strong>MNIST</strong> berisi gambar tulisan tangan angka 0-9. Setiap gambar berukuran 28×28 piksel (grayscale).</p>

<ul>
    <li>60.000 data training</li>
    <li>10.000 data testing</li>
    <li>10 kelas (angka 0-9)</li>
</ul>

<p>Arsitektur CNN yang akan kita bangun:</p>
<p><code>Input(28x28x1) → Conv2D(16) → MaxPool → Conv2D(32) → MaxPool → Flatten → Dense(64) → Output(10, softmax)</code></p>

In [ ]:
# Load dataset MNIST
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Jumlah kelas: {len(np.unique(y_train))}")
print(f"Label unik: {np.unique(y_train)}")

In [ ]:
# Visualisasi beberapa contoh gambar
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, X_train.shape[0])
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}', fontsize=12)
    ax.axis('off')
plt.suptitle('Contoh Gambar MNIST', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing
# 1. Reshape: tambahkan dimensi channel (28,28) -> (28,28,1)
X_train_cnn = X_train.reshape(-1, 28, 28, 1).astype('float32')
X_test_cnn = X_test.reshape(-1, 28, 28, 1).astype('float32')

# 2. Normalisasi piksel dari [0, 255] ke [0, 1]
X_train_cnn = X_train_cnn / 255.0
X_test_cnn = X_test_cnn / 255.0

# 3. One-hot encoding untuk label
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Shape setelah preprocessing: {X_train_cnn.shape}")
print(f"Contoh label one-hot (angka {y_train[0]}): {y_train_cat[0]}")

In [ ]:
# Membangun model CNN untuk MNIST
model_mnist = Sequential([
    # Convolutional Block 1
    Conv2D(16, (3,3), activation='relu', input_shape=(28,28,1), padding='same'),
    MaxPooling2D(2,2),

    # Convolutional Block 2
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),

    # Classifier
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model_mnist.summary()

<p><strong>Penjelasan arsitektur:</strong></p>
<ul>
    <li><strong>Conv2D(16, (3,3))</strong> — 16 filter berukuran 3×3, dengan same padding agar ukuran output tetap 28×28</li>
    <li><strong>MaxPooling2D(2,2)</strong> — Mengurangi dimensi menjadi setengahnya (28×28 → 14×14)</li>
    <li><strong>Conv2D(32, (3,3))</strong> — 32 filter berukuran 3×3, output 14×14</li>
    <li><strong>MaxPooling2D(2,2)</strong> — Mengurangi lagi (14×14 → 7×7)</li>
    <li><strong>Flatten()</strong> — Mengubah tensor 3D menjadi 1D (7×7×32 = 1568)</li>
    <li><strong>Dense(64)</strong> — Fully connected layer dengan 64 neuron</li>
    <li><strong>Dense(10, softmax)</strong> — Output layer untuk 10 kelas</li>
</ul>

In [ ]:
# Compile dan training model
model_mnist.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_mnist = model_mnist.fit(
    X_train_cnn, y_train_cat,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluasi model pada data test
test_loss, test_acc = model_mnist.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Fungsi untuk memplot training history
def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'], 'b-', label='Training')
    ax1.plot(history.history['val_accuracy'], 'r-', label='Validation')
    ax1.set_title(f'Accuracy History {title}', fontsize=13)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'], 'b-', label='Training')
    ax2.plot(history.history['val_loss'], 'r-', label='Validation')
    ax2.set_title(f'Loss History {title}', fontsize=13)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_mnist, '- MNIST CNN')

In [ ]:
# Visualisasi prediksi
predictions = model_mnist.predict(X_test_cnn[:10])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_test[i], cmap='gray')
    pred_label = np.argmax(predictions[i])
    true_label = y_test[i]
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {pred_label} | True: {true_label}',
                 color=color, fontsize=11)
    ax.axis('off')
plt.suptitle('Hasil Prediksi CNN pada MNIST (Hijau=Benar, Merah=Salah)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<h3><font color="blue">Latihan 2:</font></h3>

<ul>
    <li>Ganti dataset MNIST dengan <strong>Fashion-MNIST</strong> (<code>tensorflow.keras.datasets.fashion_mnist</code>). Apa perbedaan hasilnya? Mengapa?</li>
    <li>Coba tambahkan 1 convolutional block lagi (Conv2D + MaxPooling). Apakah akurasi meningkat?</li>
    <li>Berapa jumlah total parameter pada model yang telah ditambahkan?</li>
</ul>

<p><strong>Hint:</strong> Fashion-MNIST memiliki dimensi dan jumlah data yang sama dengan MNIST, tetapi berisi gambar item fashion (baju, sepatu, tas, dll.) sehingga lebih menantang.</p>

In [ ]:
# Kerjakan Latihan 2 pada cell berikut ini



---
## 4. CNN untuk CIFAR-10 (RGB Images)

<p>Sekarang kita akan mengklasifikasikan gambar berwarna (RGB) menggunakan dataset <strong>CIFAR-10</strong>:</p>
<ul>
    <li>50.000 data training, 10.000 data testing</li>
    <li>Gambar berukuran <strong>32×32 piksel × 3 channel (RGB)</strong></li>
    <li>10 kelas: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck</li>
</ul>

<p>Tantangan: gambar CIFAR-10 berukuran kecil dan lebih kompleks dari MNIST.</p>

In [ ]:
# Load CIFAR-10
from tensorflow.keras.datasets import cifar10

(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training shape: {X_train_cifar.shape}")
print(f"Testing shape: {X_test_cifar.shape}")

# Visualisasi
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, X_train_cifar.shape[0])
    ax.imshow(X_train_cifar[idx])
    ax.set_title(f'{class_names[y_train_cifar[idx][0]]}', fontsize=12)
    ax.axis('off')
plt.suptitle('Contoh Gambar CIFAR-10', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing CIFAR-10
X_train_c = X_train_cifar.astype('float32') / 255.0
X_test_c = X_test_cifar.astype('float32') / 255.0

y_train_c = to_categorical(y_train_cifar, 10)
y_test_c = to_categorical(y_test_cifar, 10)

print(f"Shape: {X_train_c.shape}, dtype: {X_train_c.dtype}")

In [ ]:
# CNN model untuk CIFAR-10 (tanpa regularisasi)
model_cifar = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3), padding='same'),
    MaxPooling2D(2,2),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),

    # Classifier
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model_cifar.summary()

In [ ]:
# Training model CIFAR-10 (tanpa regularisasi)
model_cifar.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_cifar = model_cifar.fit(
    X_train_c, y_train_c,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_c, y_test_c),
    verbose=1
)

# Evaluasi
test_loss, test_acc = model_cifar.evaluate(X_test_c, y_test_c, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
plot_history(history_cifar, '- CIFAR-10 (Tanpa Regularisasi)')

---
## 5. Mengatasi Overfitting

<p>Perhatikan grafik di atas. Jika <strong>training accuracy terus naik</strong> sementara <strong>validation accuracy stagnan/turun</strong>, maka model mengalami <strong>overfitting</strong>.</p>

<p><strong>Solusi Overfitting:</strong></p>
<ol>
    <li><strong>Dropout</strong> — Secara acak menonaktifkan sejumlah neuron selama training untuk mencegah model terlalu bergantung pada fitur tertentu</li>
    <li><strong>Data Augmentation</strong> — Memperbanyak variasi data training dengan transformasi (rotasi, flip, zoom, shift, dll.)</li>
    <li><strong>Menambah data training</strong></li>
    <li><strong>Early Stopping</strong> — Menghentikan training saat validation loss mulai meningkat</li>
</ol>

In [ ]:
# CNN model CIFAR-10 DENGAN Dropout
model_cifar_v2 = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3), padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Classifier
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

print("Model dengan Dropout:")
model_cifar_v2.summary()

In [ ]:
# Data Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Konfigurasi augmentasi untuk data training
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Visualisasi Data Augmentation
sample_img = X_train_c[0:1]  # ambil 1 gambar
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(sample_img[0])
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

aug_iter = train_datagen.flow(sample_img, batch_size=1)
for i in range(1, 5):
    aug_img = next(aug_iter)[0]
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Augmented {i}', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Contoh Data Augmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Training dengan Dropout + Data Augmentation
model_cifar_v2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_cifar_v2 = model_cifar_v2.fit(
    train_datagen.flow(X_train_c, y_train_c, batch_size=64),
    epochs=15,
    steps_per_epoch=len(X_train_c) // 64,
    validation_data=(X_test_c, y_test_c),
    verbose=1
)

test_loss_v2, test_acc_v2 = model_cifar_v2.evaluate(X_test_c, y_test_c, verbose=0)
print(f"\nTest Accuracy (dengan Dropout + Augmentation): {test_acc_v2:.4f}")

In [ ]:
plot_history(history_cifar_v2, '- CIFAR-10 (Dropout + Augmentation)')

In [ ]:
# Perbandingan hasil
print("=" * 50)
print("PERBANDINGAN HASIL CIFAR-10")
print("=" * 50)
print(f"CNN Tanpa Regularisasi  : {test_acc:.4f}")
print(f"CNN + Dropout + AugData : {test_acc_v2:.4f}")
print("=" * 50)

<h3><font color="blue">Latihan 3:</font></h3>

<ul>
    <li>Bandingkan grafik <strong>loss</strong> antara model tanpa regularisasi dan model dengan Dropout + Data Augmentation. Apa yang terjadi pada gap antara training loss dan validation loss?</li>
    <li>Coba ubah parameter Dropout menjadi 0.3 dan 0.7. Mana yang memberikan hasil terbaik?</li>
    <li>Tambahkan teknik <strong>Early Stopping</strong> menggunakan <code>tf.keras.callbacks.EarlyStopping</code>. Bagaimana pengaruhnya?</li>
</ul>

<p><strong>Hint Early Stopping:</strong></p>

```python
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
model.fit(..., callbacks=[early_stop])
```

In [ ]:
# Kerjakan Latihan 3 pada cell berikut ini



---
## 6. Transfer Learning

<p><strong>Transfer Learning</strong> adalah teknik menggunakan model yang sudah dilatih (<em>pre-trained</em>) pada dataset besar (seperti ImageNet) dan mengadaptasinya untuk tugas baru.</p>

<p><strong>Mengapa Transfer Learning?</strong></p>
<ul>
    <li>Deep Learning membutuhkan <strong>data besar</strong> untuk performa optimal</li>
    <li>Melatih model dari awal membutuhkan <strong>waktu dan komputasi</strong> yang sangat besar</li>
    <li>Pre-trained model sudah mempelajari fitur visual umum (edges, textures, shapes) yang bisa di-<em>transfer</em></li>
</ul>

<p><strong>Strategi Transfer Learning:</strong></p>
<ol>
    <li><strong>Feature Extraction</strong> — Freeze seluruh layer pre-trained, ganti hanya classifier (Dense layer) di akhir</li>
    <li><strong>Fine-Tuning (Partial)</strong> — Freeze sebagian besar layer, unfreeze beberapa layer terakhir untuk di-retrain</li>
    <li><strong>Fine-Tuning (Full)</strong> — Retrain seluruh layer dengan bobot pre-trained sebagai inisialisasi</li>
</ol>

### Pre-trained Models Populer:

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Model</th><th>Tahun</th><th>Input Size</th><th>Keunggulan</th></tr>
<tr><td><strong>VGG16</strong></td><td>2014</td><td>224×224</td><td>Arsitektur sederhana, mudah dipahami</td></tr>
<tr><td><strong>InceptionV3</strong></td><td>2014</td><td>299×299</td><td>Multi-scale feature extraction (inception module)</td></tr>
<tr><td><strong>ResNet50</strong></td><td>2015</td><td>224×224</td><td>Skip connections, bisa sangat dalam tanpa degradasi</td></tr>
<tr><td><strong>MobileNetV2</strong></td><td>2018</td><td>224×224</td><td>Ringan, cocok untuk mobile/edge device</td></tr>
</table>

### Transfer Learning dengan VGG16 pada CIFAR-10

<p>Kita akan menggunakan <strong>VGG16</strong> yang sudah dilatih pada ImageNet (1000 kelas, 1.2 juta gambar) sebagai feature extractor, kemudian menambahkan classifier baru untuk CIFAR-10 (10 kelas).</p>

In [ ]:
# Load VGG16 pre-trained model (tanpa layer Dense/classifier di atasnya)
from tensorflow.keras.applications import VGG16

base_model = VGG16(
    weights='imagenet',      # Bobot dari ImageNet
    include_top=False,       # Hapus fully connected layers
    input_shape=(32, 32, 3)  # Sesuaikan dengan input kita
)

# Lihat arsitektur base model
print(f"Jumlah layers VGG16: {len(base_model.layers)}")
print(f"Total parameters: {base_model.count_params():,}")
base_model.summary()

In [ ]:
# Freeze seluruh layer VGG16 (Feature Extraction mode)
for layer in base_model.layers:
    layer.trainable = False

# Bangun model lengkap: VGG16 + Custom Classifier
model_tl = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model_tl.summary()

# Cek: berapa parameter yang trainable vs frozen?
trainable = sum([tf.keras.backend.count_params(w) for w in model_tl.trainable_weights])
non_trainable = sum([tf.keras.backend.count_params(w) for w in model_tl.non_trainable_weights])
print(f"\nTrainable params: {trainable:,}")
print(f"Non-trainable (frozen) params: {non_trainable:,}")

In [ ]:
# Training Transfer Learning model
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_tl = model_tl.fit(
    X_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_c, y_test_c),
    verbose=1
)

test_loss_tl, test_acc_tl = model_tl.evaluate(X_test_c, y_test_c, verbose=0)
print(f"\nTest Accuracy (VGG16 Transfer Learning): {test_acc_tl:.4f}")

In [ ]:
plot_history(history_tl, '- VGG16 Transfer Learning (Frozen)')

<h3><font color="blue">Latihan 4:</font></h3>

<ul>
    <li>Mengapa pada kasus CIFAR-10 (32×32), hasil VGG16 mungkin tidak sebaik yang diharapkan? (Hint: VGG16 di-desain untuk gambar 224×224)</li>
    <li>Coba resize gambar CIFAR-10 ke ukuran <strong>64×64</strong> atau <strong>96×96</strong> menggunakan <code>tf.image.resize()</code>. Apakah hasilnya lebih baik?</li>
</ul>

<p><strong>Hint resize:</strong></p>

```python
X_train_resized = tf.image.resize(X_train_c, (96, 96)).numpy()
X_test_resized = tf.image.resize(X_test_c, (96, 96)).numpy()
# Jangan lupa ubah input_shape di VGG16 juga
```

In [ ]:
# Kerjakan Latihan 4 pada cell berikut ini



---
## 7. Fine-Tuning Pre-trained Model

<p>Pada bagian sebelumnya, kita membekukan (<em>freeze</em>) seluruh layer VGG16. Sekarang kita akan <strong>unfreeze beberapa layer terakhir</strong> agar model bisa menyesuaikan fitur high-level dengan dataset kita.</p>

<p><strong>Strategi:</strong> Unfreeze layer-layer di block terakhir (block5) sementara layer awal tetap frozen. Layer awal sudah mempelajari fitur umum (edges, textures) yang berguna untuk banyak task.</p>

In [ ]:
# Lihat nama semua layer di VGG16
for i, layer in enumerate(base_model.layers):
    print(f"Layer {i}: {layer.name}, trainable={layer.trainable}")

In [ ]:
# Fine-tuning: Unfreeze block5 (4 layer terakhir dari conv layers)
for layer in base_model.layers[-4:]:
    layer.trainable = True

print("Status layer setelah unfreeze block5:")
for i, layer in enumerate(base_model.layers):
    print(f"  Layer {i}: {layer.name:25s} trainable={layer.trainable}")

In [ ]:
# Re-compile dengan learning rate kecil (penting untuk fine-tuning!)
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # LR lebih kecil
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tune training
history_ft = model_tl.fit(
    X_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_c, y_test_c),
    verbose=1
)

test_loss_ft, test_acc_ft = model_tl.evaluate(X_test_c, y_test_c, verbose=0)
print(f"\nTest Accuracy (VGG16 Fine-Tuned): {test_acc_ft:.4f}")

In [ ]:
plot_history(history_ft, '- VGG16 Fine-Tuned')

---
## 8. Perbandingan dengan Model Pre-trained Lain: MobileNetV2

<p><strong>MobileNetV2</strong> adalah arsitektur yang lebih ringan dan efisien dibanding VGG16, dirancang untuk perangkat mobile. Mari kita bandingkan performanya.</p>

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SECTION 8 — SETUP MANDIRI (Self-Contained)                  ║
# ║  Jalankan cell ini PERTAMA sebelum cell lain di Section 8.   ║
# ║  Cell ini membersihkan memori lama & memuat ulang semua       ║
# ║  kebutuhan dari awal agar RAM tidak penuh.                   ║
# ╚══════════════════════════════════════════════════════════════╝

import gc

# ── 1. BERSIHKAN MODEL LAMA DARI MEMORI ─────────────────────────────────────
# Hapus model-model sebelumnya agar RAM bebas
for var in ['model_mnist', 'model_cifar', 'model_cifar_v2',
            'model_tl', 'base_model',
            'history_mnist', 'history_cifar', 'history_cifar_v2',
            'history_tl', 'history_ft']:
    if var in dir():
        del globals()[var]
        print(f'[BERSIH] {var} dihapus dari memori')

gc.collect()  # paksa garbage collector bekerja
print('[INFO] Memori lama dibersihkan.\n')

# ── 2. IMPORT SEMUA LIBRARY ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.datasets import cifar10

print(f'[INFO] TensorFlow {tf.__version__} siap.')
print(f'[INFO] GPU: {tf.config.list_physical_devices("GPU")}')

# ── 3. LOAD & PREPROCESS CIFAR-10 ───────────────────────────────────────────
print('\n[INFO] Memuat CIFAR-10...')
(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

X_train_c = X_train_cifar.astype('float32') / 255.0
X_test_c  = X_test_cifar.astype('float32')  / 255.0
y_train_c = to_categorical(y_train_cifar, 10)
y_test_c  = to_categorical(y_test_cifar,  10)

print(f'[INFO] X_train_c : {X_train_c.shape}')
print(f'[INFO] X_test_c  : {X_test_c.shape}')

# ── 4. RESIZE KE 96x96 UNTUK MOBILENETV2 ────────────────────────────────────
print('\n[INFO] Resize gambar ke 96x96...')
X_train_resized = tf.image.resize(X_train_c, (96, 96)).numpy()
X_test_resized  = tf.image.resize(X_test_c,  (96, 96)).numpy()

print(f'[INFO] X_train_resized : {X_train_resized.shape}')
print(f'[INFO] X_test_resized  : {X_test_resized.shape}')

# ── 5. DEFINISI FUNGSI HELPER ────────────────────────────────────────────────
def plot_history(history, title=''):
    """Plot accuracy & loss dari training history."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['accuracy'],     'b-', label='Training',   linewidth=2)
    ax1.plot(history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
    ax1.set_title(f'Accuracy History {title}', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'],     'b-', label='Training',   linewidth=2)
    ax2.plot(history.history['val_loss'], 'r-', label='Validation', linewidth=2)
    ax2.set_title(f'Loss History {title}', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print('\n[INFO] Fungsi plot_history siap.')
print('\n✅ Setup Section 8 selesai. Lanjutkan ke cell berikutnya.')


In [ ]:
# Load MobileNetV2
base_mobilenet = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze semua layer
base_mobilenet.trainable = False

# Bangun model
model_mobile = Sequential([
    base_mobilenet,
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model_mobile.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mobile.summary()

In [ ]:
# Training MobileNetV2
history_mobile = model_mobile.fit(
    X_train_resized, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_resized, y_test_c),
    verbose=1
)

test_loss_m, test_acc_m = model_mobile.evaluate(X_test_resized, y_test_c, verbose=0)
print(f"\nTest Accuracy (MobileNetV2): {test_acc_m:.4f}")

In [ ]:
# Definisi fungsi plot_history (jaga-jaga jika cell sebelumnya belum dijalankan)
if 'plot_history' not in dir():
    def plot_history(history, title=''):
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(history.history['accuracy'], 'b-', label='Training')
        ax1.plot(history.history['val_accuracy'], 'r-', label='Validation')
        ax1.set_title(f'Accuracy History {title}', fontsize=13)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(history.history['loss'], 'b-', label='Training')
        ax2.plot(history.history['val_loss'], 'r-', label='Validation')
        ax2.set_title(f'Loss History {title}', fontsize=13)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

plot_history(history_mobile, '- MobileNetV2 Transfer Learning')

In [ ]:
plot_history(history_mobile, '- MobileNetV2 Transfer Learning')

In [ ]:
# Ringkasan perbandingan semua model
# Gunakan .get() pattern agar tidak error jika salah satu model belum dijalankan

results = {
    #"1. CNN Sederhana (MNIST)"            : locals().get('test_acc_mnist', None),
    #"2. CNN Tanpa Regularisasi (CIFAR-10)": locals().get('test_acc',       None),
    #"3. CNN + Dropout + Augmentation"     : locals().get('test_acc_v2',    None),
    #"4. VGG16 Transfer Learning (Frozen)" : locals().get('test_acc_tl',    None),
    #"5. VGG16 Fine-Tuned (Block5)"        : locals().get('test_acc_ft',    None),
    "6. MobileNetV2 Transfer Learning"    : locals().get('test_acc_m',     None),
}

print("=" * 60)
print("RINGKASAN PERBANDINGAN SEMUA MODEL")
print("=" * 60)
print(f"{'Model':<40} {'Test Acc':>10}")
print("-" * 60)
for nama, acc in results.items():
    nilai = f"{acc:.4f}" if acc is not None else "belum dijalankan"
    print(f"{nama:<40} {nilai:>10}")
print("=" * 60)

<h3><font color="blue">Latihan 5:</font></h3>

<ul>
    <li>Ganti pre-trained model dengan <strong>ResNet50</strong> (<code>tf.keras.applications.ResNet50</code>). Bandingkan hasilnya dengan VGG16 dan MobileNetV2.</li>
    <li>Mengapa kita menggunakan <code>GlobalAveragePooling2D()</code> pada MobileNetV2 daripada <code>Flatten()</code>? Apa kelebihannya?</li>
    <li>Coba lakukan <strong>Fine-Tuning</strong> pada MobileNetV2 (unfreeze 20 layer terakhir) dengan learning rate yang kecil. Apakah hasilnya meningkat?</li>
</ul>

<p><strong>Hint ResNet50:</strong></p>

```python
from tensorflow.keras.applications import ResNet50
base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
```

In [ ]:
# Kerjakan Latihan 5 pada cell berikut ini



---
## 9. Prediksi pada Gambar Baru

<p>Setelah model dilatih, kita bisa menggunakannya untuk memprediksi gambar baru.</p>

In [ ]:
# Prediksi pada data test dan visualisasi
# Menggunakan model_mobile (MobileNetV2) karena model_cifar_v2
# tidak tersedia setelah RAM dibersihkan di Section 8.

# Pastikan data tersedia
if 'X_test_c' not in dir():
    from tensorflow.keras.datasets import cifar10
    from tensorflow.keras.utils import to_categorical
    (_, _), (X_test_cifar, y_test_cifar) = cifar10.load_data()
    X_test_c = X_test_cifar.astype('float32') / 255.0
    y_test_c = to_categorical(y_test_cifar, 10)
    class_names = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

# Gunakan model terbaik yang tersedia di memori
if 'model_mobile' in dir():
    active_model = model_mobile
    model_name   = 'MobileNetV2 Transfer Learning'
    # MobileNetV2 butuh input 96x96
    X_pred = tf.image.resize(X_test_c[:20], (96, 96)).numpy()
else:
    raise NameError('Jalankan cell training MobileNetV2 di Section 8 terlebih dahulu!')

predictions = active_model.predict(X_pred, verbose=0)

fig, axes = plt.subplots(4, 5, figsize=(16, 12))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_test_cifar[i])
    pred_label = class_names[np.argmax(predictions[i])]
    true_label = class_names[y_test_cifar[i][0]]
    confidence = np.max(predictions[i]) * 100
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {pred_label} ({confidence:.1f}%)\nTrue: {true_label}',
                 color=color, fontsize=10)
    ax.axis('off')

plt.suptitle(f'Hasil Prediksi {model_name} pada CIFAR-10\n(Hijau = Benar, Merah = Salah)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 10. Bonus: Prediksi Gambar dari Internet

<p>Kita bisa mendownload gambar dari internet dan memprediksinya menggunakan pre-trained VGG16 (full model, 1000 kelas ImageNet).</p>

In [ ]:
# Prediksi gambar dari internet menggunakan VGG16 full (1000 kelas ImageNet)
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import urllib.request

# Load VGG16 full model
model_full = VGG16(weights='imagenet')

def predict_image_from_url(url, model):
    """Download gambar dari URL dan prediksi menggunakan VGG16"""
    path = '/tmp/test_image.jpg'

    # Tambahkan User-Agent agar tidak diblokir server (fix HTTPError 403)
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        with open(path, 'wb') as f:
            f.write(response.read())

    # Load dan preprocess
    img = image.load_img(path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # Prediksi
    preds = model.predict(x, verbose=0)
    results = decode_predictions(preds, top=3)[0]

    # Tampilkan
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    title = '\n'.join([f'{name}: {score*100:.1f}%' for (_, name, score) in results])
    plt.title(f'Top-3 Predictions:\n{title}', fontsize=11)
    plt.axis('off')
    plt.show()

# Gunakan URL dari sumber yang tidak memblokir Python requests
predict_image_from_url(
    'https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg',
    model_full
)

<h3><font color="blue">Latihan 6 (Tugas):</font></h3>

<ul>
    <li>Cari 3 gambar objek berbeda dari internet dan prediksi menggunakan VGG16 full model. Apakah prediksinya akurat?</li>
    <li>Coba ganti model menjadi <strong>ResNet50</strong> atau <strong>MobileNetV2</strong> untuk prediksi yang sama. Bandingkan hasilnya.</li>

In [ ]:
# Kerjakan Latihan 6 pada cell berikut ini



---
## Rangkuman

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Topik</th><th>Poin Penting</th></tr>
<tr><td><strong>CNN</strong></td><td>Arsitektur neural network untuk data visual; terdiri dari Conv, Pooling, Dense layers; belajar fitur secara hierarkis</td></tr>
<tr><td><strong>Overfitting</strong></td><td>Terjadi saat model terlalu menghafal data training; solusi: Dropout, Data Augmentation, Early Stopping</td></tr>
<tr><td><strong>Transfer Learning</strong></td><td>Menggunakan model pre-trained untuk task baru; sangat berguna saat data terbatas</td></tr>
<tr><td><strong>Fine-Tuning</strong></td><td>Membuka (unfreeze) sebagian layer pre-trained untuk disesuaikan; gunakan learning rate kecil</td></tr>
</table>

---
<center><strong>Selamat Mengerjakan!</strong></center>
